# 📊 Evaluate LLM-JEPA on AI Feynman Benchmark

This notebook evaluates a trained LLM-JEPA model on the AI Feynman symbolic regression benchmark.

**What this does:**
- Clones/pulls the repository
- Syncs to Google Drive (SymbolicRegression folder)
- Downloads AI Feynman dataset if needed
- Runs comprehensive evaluation on all 100 equations
- Generates detailed metrics report

**Runtime:** T4 GPU recommended (for BFGS constant fitting)

---

In [ ]:
# @title 📦 Setup: Clone Repo & Sync to Google Drive

import os
import subprocess
import yaml
from pathlib import Path
from google.colab import drive

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Define paths
DRIVE_FOLDER = "/content/drive/MyDrive/SymbolicRegression"
WORK_DIR = "/content/GSOC-LM-JEPA_for_Symbolic_Regression"
REPO_URL = "https://github.com/udohchuks/GSOC-LM-JEPA_for_Symbolic_Regression.git"

# Create Drive folder if not exists
Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
print(f"✅ Drive folder ready: {DRIVE_FOLDER}")

# Clone or pull repository
if os.path.exists(WORK_DIR):
    print("📦 Repository found, pulling latest changes...")
    os.chdir(WORK_DIR)
    subprocess.run(["git", "pull"], check=True)
else:
    print("📦 Cloning repository...")
    os.chdir("/content")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(WORK_DIR)

# Sync to Drive (copy working directory)
print("🔄 Syncing to Google Drive...")
sync_target = f"{DRIVE_FOLDER}/code"
Path(sync_target).mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-av", "--delete", f"{WORK_DIR}/", f"{sync_target}/"], check=True)
print(f"✅ Synced to: {sync_target}")

# Install dependencies
print("📦 Installing dependencies...")
os.chdir(WORK_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Dependencies installed")

In [ ]:
# @title 📥 Download AI Feynman Dataset (if needed)

import tarfile
import urllib.request
import shutil
from pathlib import Path

# Setup AI Feynman data in Drive
print("📦 Setting up AI Feynman dataset in Drive...")
local_data = Path(f"{WORK_DIR}/data/Feynman_with_units")
drive_data = Path(f"{DRIVE_FOLDER}/Feynman_with_units")

if not drive_data.exists():
    if local_data.exists():
        print(f"   📥 Copying from local: {local_data}")
        shutil.copytree(local_data, drive_data)
        print(f"   ✅ Copied to Drive: {drive_data}")
    else:
        print("   ⚠️  Local AI Feynman data not found!")
        print("   Downloading from Dropbox...")
        
        # Download
        tar_path = Path(f"{DRIVE_FOLDER}/Feynman_with_units.tar.gz")
        if not tar_path.exists():
            url = "https://www.dropbox.com/s/7kgfr00qpokgz8w/Feynman_with_units.tar.gz?dl=1"
            print(f"   Downloading: {url}")
            urllib.request.urlretrieve(url, tar_path)
            print(f"   ✅ Downloaded: {tar_path}")
        
        # Extract
        print(f"   Extracting to: {drive_data}")
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(path=DRIVE_FOLDER)
        print(f"   ✅ Extracted to: {drive_data}")
else:
    print(f"   ✅ AI Feynman data already in Drive: {drive_data}")

# Update config to use Drive paths
import yaml
with open(f"{WORK_DIR}/configs/small.yaml", 'r') as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = str(drive_data) + '/'
config['data']['csv_path'] = str(Path(DRIVE_FOLDER) / 'FeynmanEquations.csv')

# Copy CSV if needed
local_csv = Path(f"{WORK_DIR}/data/FeynmanEquations.csv")
drive_csv = Path(config['data']['csv_path'])
if not drive_csv.exists() and local_csv.exists():
    shutil.copy(local_csv, drive_csv)
    print(f"   ✅ Copied CSV to Drive: {drive_csv}")

with open(f"{WORK_DIR}/configs/small.yaml", 'w') as f:
    yaml.dump(config, f)

print(f"✅ Config updated:")
print(f"   data_dir: {config['data']['data_dir']}")
print(f"   csv_path: {config['data']['csv_path']}")

data_dir = Path(f"{DRIVE_FOLDER}/Feynman_with_units")

if data_dir.exists() and len(list(data_dir.glob("*"))) > 10:
    print(f"✅ AI Feynman dataset already exists: {data_dir}")
    print(f"   Found {len(list(data_dir.glob('*')))} files")
    except Exception as e:
        print(f"⚠️ Download failed: {e}")
        print("   Please download manually from the repository")

In [ ]:
# @title ⚙️ Evaluation Configuration

# @markdown ### Checkpoint Selection
CHECKPOINT_PATH = ""  # @param {type: "string"}
# @markdown Leave empty to use latest checkpoint from Drive

# @markdown ### Evaluation Parameters
N_CANDIDATES = 10  # @param {type: "integer"}
TEMPERATURE = 0.8  # @param {type: "number"}
N_RESTARTS = 3  # @param {type: "integer"}

# @markdown ### Output
OUTPUT_DIR = f"{DRIVE_FOLDER}/results"  # @param {type: "string"}

# Find latest checkpoint if not specified
if not CHECKPOINT_PATH:
    ckpt_dir = Path(f"{DRIVE_FOLDER}/checkpoints")
    if ckpt_dir.exists():
        ckpts = sorted(ckpt_dir.glob("*.ckpt"), key=lambda x: x.stat().st_mtime, reverse=True)
        if ckpts:
            CHECKPOINT_PATH = str(ckpts[0])
            print(f"📁 Using latest checkpoint: {CHECKPOINT_PATH}")
        else:
            print("⚠️ No checkpoints found. Please train first or specify path.")
    else:
        print("⚠️ Checkpoint directory not found. Please train first or specify path.")

# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"✅ Output directory: {OUTPUT_DIR}")

print(f"\n📋 Evaluation settings:")
print(f"   - Checkpoint: {CHECKPOINT_PATH}")
print(f"   - Candidates: {N_CANDIDATES}")
print(f"   - Temperature: {TEMPERATURE}")
print(f"   - BFGS restarts: {N_RESTARTS}")

In [ ]:
# @title 🚀 Run Full Evaluation

import time
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
eval_output = f"{OUTPUT_DIR}/eval_{timestamp}"

print(f"🎯 Starting evaluation...")
print(f"   Checkpoint: {CHECKPOINT_PATH}")
print(f"   Output: {eval_output}")
print("=" * 60)

start_time = time.time()

# Run evaluation
%cd $WORK_DIR
!python -m run_eval --ckpt {CHECKPOINT_PATH} --mode eval --output_dir {eval_output} --n_candidates {N_CANDIDATES} --temperature {TEMPERATURE}

elapsed = time.time() - start_time
hours = elapsed / 3600

print("\n" + "=" * 60)
print(f"✅ Evaluation complete!")
print(f"   Time elapsed: {hours:.2f} hours ({elapsed:.0f} seconds)")
print(f"   Results saved to: {eval_output}")

In [ ]:
# @title 📊 View Results Summary

import json
from pathlib import Path

# Find latest evaluation results
results_dir = Path(eval_output)
metrics_file = results_dir / "metrics.json"
report_file = results_dir / "evaluation_report.md"

if metrics_file.exists():
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    
    print("📊 Evaluation Summary")
    print("=" * 60)
    print(f"Equations evaluated: {metrics.get('n_equations', 'N/A')}")
    print(f"Exact recovery rate: {metrics.get('exact_recovery_rate', 0)*100:.1f}%")
    print(f"Valid RPN rate: {metrics.get('valid_rpn_rate', 0)*100:.1f}%")
    print(f"Dimensionally valid: {metrics.get('dim_valid_rate', 0)*100:.1f}%")
    print()
    print(f"Mean R² (pre-BFGS): {metrics.get('mean_r2_pre_bfgs', 0):.4f}")
    print(f"Mean R² (post-BFGS): {metrics.get('mean_r2_post_bfgs', 0):.4f}")
    print()
    print(f"Mean node count: {metrics.get('mean_node_count', 0):.1f}")
    print(f"Mean generation time: {metrics.get('mean_latency_generate_s', 0)*1000:.1f} ms")
    print(f"Mean BFGS time: {metrics.get('mean_latency_bfgs_s', 0)*1000:.1f} ms")
    print("=" * 60)
    
    # Recovery by variable count
    if 'recovery_by_n_vars' in metrics:
        print("\n📈 Recovery by Variable Count:")
        for n_vars, rate in sorted(metrics['recovery_by_n_vars'].items()):
            print(f"   {n_vars} variables: {rate*100:.1f}%")
    
    print(f"\n📄 Full report: {report_file}")
    
    # Display report if exists
    if report_file.exists():
        print("\n" + "=" * 60)
        print("📋 EVALUATION REPORT PREVIEW")
        print("=" * 60)
        with open(report_file, 'r') as f:
            print(f.read()[:2000])  # First 2000 chars
            print("\n... (see full report in Drive)")
else:
    print("⚠️ Metrics file not found. Check evaluation output.")

In [ ]:
# @title 🔍 Test Single Equation (Optional)

# @markdown Test on a specific AI Feynman equation
EQUATION_ID = "I.6.2a"  # @param {type: "string"}

print(f"🧪 Testing on equation: {EQUATION_ID}")
print("=" * 60)

%cd $WORK_DIR
!python -m run_eval --ckpt {CHECKPOINT_PATH} --mode predict --id {EQUATION_ID} --n_candidates {N_CANDIDATES} --temperature {TEMPERATURE}

---
## Metrics Explained

### Primary Metrics
- **Exact Recovery Rate**: % of equations recovered exactly (matches ground truth)
- **Mean R² (post-BFGS)**: Average R² after constant fitting (higher is better)
- **Valid RPN Rate**: % of generated formulas with valid Reverse Polish Notation
- **Dimensional Validity**: % of formulas that are dimensionally consistent

### Robustness Metrics
- **Noise Tolerance**: R² under increasing noise levels (ε = 0.001, 0.01, 0.1)
- **Data Efficiency**: R² with limited data points (N = 10, 50, 100, 200)
- **Extrapolation**: R² on out-of-distribution data

### Operational Metrics
- **Generation Latency**: Time to generate formula (ms)
- **BFGS Time**: Time for constant fitting (ms)

## Tips

- **Higher N_CANDIDATES**: Better recovery but slower (try 10-20)
- **Temperature**: Higher = more diverse candidates (0.7-0.9 recommended)
- **Results saved**: All results persist in `SymbolicRegression/results/`